# Purpose / このノートブックの目的

**EN**

This notebook establishes a minimal baseline for the mental health risk screening task.

The objective is to assess whether the task is sufficiently non-trivial to require supervised modeling.

To do so, we evaluate a majority-class (dummy) classifier as a reference point for performance.

The results determine whether more expressive models are justified.


**JP**

本ノートブックでは、
メンタルヘルスリスクスクリーニング問題に対する最小限のベースラインを構築する。

目的は、本タスクが教師あり学習を必要とする非自明な問題であるかを評価することである。

その基準として、多数派クラスのみを予測するダミーモデルを評価する。

結果に基づき、より表現力の高いモデルの必要性を判断する。

## 0. Imports & Setup / インポートと初期設定

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

## 1. Data loading & target definition

In [3]:
df = pd.read_csv("../data/raw/social_media_mental_health.csv")

df = df.copy()
df["target"] = (df["PHQ_9_Score"] >= 10).astype(int)

LEAK_COLS = [
    "PHQ_9_Score",
    "PHQ_9_Severity",
    "GAD_7_Score",
    "GAD_7_Severity"
]

X = df.drop(columns=LEAK_COLS + ["target", "User_ID"])
y = df["target"]

**EN**

The dataset is loaded and a binary target is constructed from `PHQ-9 ≥ 10`.

Because the target is directly derived from PHQ-9 score, all PHQ-9 / GAD-7 scores and severity labels are excluded to prevent label leakage.

User_ID is also removed as it carries no predictive value.


**JP**

データを読み込み、`PHQ-9 ≥ 10` を基準として2値の目的変数を生成する。

目的変数は PHQ-9 スコアから直接生成されるため、PHQ-9 / GAD-7 のスコアおよび重症度ラベルはラベルリーケージ防止のため除外する。

User_ID は予測情報を持たないため削除する。


## 2. Train–Test Split / 学習・評価データ分割

**EN**

We split the data into training and test sets (80/20).

Stratified sampling is applied to preserve the class imbalance in both splits, ensuring consistent minority representation.

The test set remains fully held out and is used only for final evaluation.


**JP**

データを 80/20 で学習用と評価用に分割する。

クラス不均衡を維持するために stratify を適用し、両データセットで少数クラス比率が一致するようにする。

テストデータは完全にホールドアウトとし、最終評価のみに使用する。

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=17,
    stratify=y
)

print("Train target rate:", round(y_train.mean(), 3))
print("Test  target rate:", round(y_test.mean(), 3))

Train target rate: 0.209
Test  target rate: 0.209


## 3. Evaluation Metrics / 評価指標

**EN**

We define a unified evaluation function to ensure consistent and comparable metric reporting across models.

Given the moderate class imbalance (~21% positive class) and the screening objective, evaluation emphasizes:

- PR-AUC (minority-class sensitivity)
- Recall (risk detection rate)
- F1-score (balance between precision and recall)

ROC-AUC is also reported for completeness.


**JP**

モデル間で一貫した比較を行うため、共通の評価関数を定義する。

陽性クラスが約21%と中程度の不均衡であり、スクリーニング用途を想定しているため、以下を重視する：

- PR-AUC（少数クラス感度）
- Recall（リスク検出率）
- F1-score（PrecisionとRecallのバランス）

ROC-AUC も参考指標として算出する。

In [11]:
def evaluate_binary_classifier(y_true, y_pred, y_proba, name: str):
    metrics = {
        "model": name,
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    return metrics

## 4. Baseline: Dummy Classifier / ダミーベースライン

**EN**

This model always predicts the majority (non-risk) class.

It serves as a naive majority-class baseline, establishing a reference performance level that any meaningful model should exceed.

**JP**

常に多数派（non-risk）クラスを予測する単純なベースラインモデルである。

意味のあるモデルであれば、この基準性能を上回る必要がある。

In [12]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)

y_pred = dummy.predict(X_test)
y_proba = dummy.predict_proba(X_test)[:, 1]

dummy_metrics = evaluate_binary_classifier(
    y_test, y_pred, y_proba, "Dummy (Most Frequent)"
)

print(dummy_metrics)
print(classification_report(y_test, y_pred, digits=3, zero_division=0))

{'model': 'Dummy (Most Frequent)', 'roc_auc': 0.5, 'pr_auc': 0.209375, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
              precision    recall  f1-score   support

           0      0.791     1.000     0.883      1265
           1      0.000     0.000     0.000       335

    accuracy                          0.791      1600
   macro avg      0.395     0.500     0.442      1600
weighted avg      0.625     0.791     0.698      1600



**EN**

As expected:

- ROC-AUC = 0.5 (no discriminative ability)
- PR-AUC ≈ positive class prevalence (~0.21)
- Recall = 0 (no risk cases detected)
- Accuracy ≈ 79% (majority-class rate)

This confirms that accuracy alone is not informative under class imbalance.


**JP**

予想通り：

- ROC-AUC = 0.5（識別能力なし）
- PR-AUC ≈ 陽性率（約0.21）
- Recall = 0（リスク群を検出できない）
- Accuracy ≈ 79%（多数派比率）

この結果は、不均衡データにおいて Accuracy 単独では
十分な指標にならないことを示している。

## 5. Interpretation of baseline results

**EN**

Conclusion:

Majority-class prediction achieves high accuracy but completely fails to detect risk cases.

A screening model must prioritize minority detection performance.


**JP**

結論：

多数派予測は高いAccuracyを示すが、リスク者の検出には失敗する。

スクリーニング用途では少数クラス検出性能が不可欠である。

## 6. Decision & Next Step / 判断と次のステップ

**EN**

Decision:

The majority-class baseline fails to detect risk cases, making it unsuitable for screening.

We therefore proceed to supervised modeling approaches.

Next notebook: `03_modeling.ipynb`

- Logistic Regression (interpretable reference model) (Not implemented yet)
- LightGBM (primary performance-oriented model)
- Threshold evaluation with emphasis on recall and PR-AUC

**JP**

判断：

多数派ベースラインはリスク者を検出できず、スクリーニング用途には適さない。

そのため、教師あり学習モデルへ進む。

次のノートブック：`03_modeling.ipynb`

- ロジスティック回帰（解釈性確認用モデル）(未着手)
- LightGBM（性能重視の主要モデル）
- Recall・PR-AUC を重視した意思決定閾値の検討
